# 问卷数据探索

快速 EDA：从 SQLite 加载两份调查数据，查看分布和缺失情况。

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.unicode_minus'] = False

from pathlib import Path
ROOT = Path('..').resolve()
print('Project root:', ROOT)

In [ ]:
def load_survey(survey_id):
    db = ROOT / 'data' / 'db' / f'{survey_id}.db'
    conn = sqlite3.connect(db)
    df = pd.read_sql_query(f"SELECT * FROM respondents WHERE survey='{survey_id}'", conn)
    conn.close()
    return df

s1 = load_survey('survey1')
s2 = load_survey('survey2')
print(f'survey1: {s1.shape}   survey2: {s2.shape}')

In [ ]:
print('=== survey1 缺失值 ===')
print(s1.isnull().sum()[s1.isnull().sum() > 0])
print('\n=== survey2 缺失值 ===')
print(s2.isnull().sum()[s2.isnull().sum() > 0])

In [ ]:
# Likert 分布
likert_cols = ['ai_accept','meta_accept','green_accept','second_accept']
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes[0], likert_cols):
    s1[col].dropna().astype(int).value_counts().sort_index().plot.bar(ax=ax, color='steelblue')
    ax.set_title(f'S1 {col}', fontsize=8)
for ax, col in zip(axes[1], likert_cols):
    s2[col].dropna().astype(int).value_counts().sort_index().plot.bar(ax=ax, color='darkorange')
    ax.set_title(f'S2 {col}', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 人口学分布
for df, name in [(s1, 'survey1'), (s2, 'survey2')]:
    print(f'\n=== {name} ===')
    for col in ['gender','age_group','status']:
        if col in df.columns:
            print(df[col].value_counts().to_string())
            print()

In [ ]:
# 数值变量描述统计
num_cols = ['impact_num','extra_spend','saving_amt','attention_num','dynamic_num']
print('=== survey1 ===')
print(s1[num_cols].describe().round(2))
print('\n=== survey2 ===')
print(s2[[c for c in num_cols if c in s2.columns]].describe().round(2))